# Do Focus forecast-error distributions change over time?

This notebook compares the **signed error distributions** of constructed 3-, 6-, and 12-month Focus IPCA forecasts across historical periods and rolling windows. For each forecast month and horizon, signed error is `forecast − realized IPCA`, in percentage points (p.p.). A **negative** error means Focus underpredicted inflation; a positive error means it overpredicted.

The three forecasts compound monthly Focus medians from the same month-end snapshot. They are matched to realized IPCA over the same windows. The constructed 12-month forecast is distinct from the BCB's separately published 12-month median. The source panel is produced by [the rolling-horizon comparison notebook](ipca_rolling_horizon_comparison.ipynb); [local data instructions](data/README.md) explain how to rebuild it. Underlying sources: [BCB Focus data](https://dadosabertos.bcb.gov.br/dataset/expectativas-mercado) and [IBGE IPCA index](https://sidra.ibge.gov.br/tabela/1737).

The question is whether the **center, tails, and spread** of the error distribution change over time *within each forecast horizon*. Raw error scales differ across horizons because the forecast windows differ.

## 1. Load the matched forecast panel

Run from the repository root or `projects/`. The panel has one row per forecast month from November 2001 through January 2025, with all three horizons matched on the same 279 dates. The rolling plots use all 279 dates. Period comparisons use complete calendar years 2002–2024.

In [ ]:
panel_file <- file.path("projects", "data", "ipca_rolling_horizon_comparison_200111_202501.csv")
project_root <- if (file.exists(panel_file)) {
  "."
} else if (file.exists(file.path("..", panel_file))) {
  ".."
} else {
  stop("Missing matched panel; run projects/ipca_rolling_horizon_comparison.ipynb first")
}
panel <- read.csv(file.path(project_root, panel_file), stringsAsFactors = FALSE)
horizons <- c(3L, 6L, 12L)
error_columns <- paste0("error_", horizons, "m_pp")
forecast_columns <- paste0("forecast_", horizons, "m_pct")
realized_columns <- paste0("realized_", horizons, "m_pct")
stopifnot(all(c("forecast_month", error_columns, forecast_columns,
                realized_columns) %in% names(panel)))
expected_months <- format(seq(as.Date("2001-11-01"),
                              as.Date("2025-01-01"), by = "month"), "%Y-%m")
stopifnot(identical(panel$forecast_month, expected_months), nrow(panel) == 279L,
          all(is.finite(as.matrix(panel[error_columns]))))
for (h in horizons) {
  error <- panel[[paste0("error_", h, "m_pp")]]
  reconstructed <- panel[[paste0("forecast_", h, "m_pct")]] -
    panel[[paste0("realized_", h, "m_pct")]]
  stopifnot(isTRUE(all.equal(error, reconstructed, tolerance = 1e-8)))
}
panel$date <- as.Date(paste0(panel$forecast_month, "-01"))
cat("Matched forecast months:", nrow(panel), "|",
    panel$forecast_month[1], "to", tail(panel$forecast_month, 1), "\n")

## 2. Compare complete-year historical periods

Use 2002–2009 (96 months), 2010–2017 (96), and 2018–2024 (84). The first two months of the panel and January 2025 are excluded **only** from the period comparison. Every included year has all 12 forecast months. Report bias (mean signed error), spread, median, mean absolute error, and the share of underpredictions.

In [ ]:
period_labels <- c("2002–2009", "2010–2017", "2018–2024")
period_colors <- c("steelblue4", "darkorange2", "darkgreen")
year <- as.integer(substr(panel$forecast_month, 1, 4))
period_panel <- panel[year >= 2002 & year <= 2024, , drop = FALSE]
period_panel$period <- cut(as.integer(substr(period_panel$forecast_month, 1, 4)),
                           breaks = c(2001, 2009, 2017, 2024),
                           labels = period_labels)
stopifnot(identical(as.integer(table(period_panel$period)), c(96L, 96L, 84L)))

errors_by_horizon <- lapply(error_columns, function(column) {
  split(period_panel[[column]], period_panel$period)
})
names(errors_by_horizon) <- paste0(horizons, "m")
distribution_summary <- do.call(rbind, lapply(seq_along(horizons), function(i) {
  do.call(rbind, lapply(period_labels, function(period) {
    x <- errors_by_horizon[[i]][[period]]
    data.frame(horizon_months = horizons[i], period = period, months = length(x),
               bias_pp = mean(x), sd_pp = sd(x), median_pp = median(x),
               p10_pp = unname(quantile(x, 0.10)),
               p90_pp = unname(quantile(x, 0.90)),
               mae_pp = mean(abs(x)),
               underprediction_pct = 100 * mean(x < 0))
  }))
}))
row.names(distribution_summary) <- NULL
print(distribution_summary, row.names = FALSE, digits = 4)

**Validated local results** (all errors in percentage points):

| Horizon | Period | Months | Mean error | SD | Median error | Underpredicted |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| 3 months | 2002–2009 | 96 | -0.204 | 0.792 | -0.096 | 63.5% |
| 3 months | 2010–2017 | 96 | -0.087 | 0.392 | -0.090 | 55.2% |
| 3 months | 2018–2024 | 84 | -0.204 | 0.660 | -0.142 | 60.7% |
| 6 months | 2002–2009 | 96 | -0.552 | 1.844 | -0.292 | 64.6% |
| 6 months | 2010–2017 | 96 | -0.215 | 0.842 | -0.203 | 57.3% |
| 6 months | 2018–2024 | 84 | -0.546 | 1.383 | -0.453 | 66.7% |
| 12 months | 2002–2009 | 96 | -1.201 | 3.403 | -0.356 | 59.4% |
| 12 months | 2010–2017 | 96 | -0.619 | 1.642 | -0.546 | 71.9% |
| 12 months | 2018–2024 | 84 | -1.263 | 2.657 | -0.258 | 61.9% |

The middle period has the lowest error SD at every horizon in this descriptive grouping. Mean signed errors are negative in all nine horizon-period combinations.

## 3. Compare signed-error histograms

Each row is one forecast horizon. Within a row, the three eras use **identical bin edges and y-axis limits**; density scaling makes the shapes comparable despite the shorter last period. The vertical dashed line marks zero error. The bin widths are 0.25, 0.5, and 1 p.p. for 3, 6, and 12 months, respectively.

In [ ]:
bin_widths <- c(0.25, 0.5, 1.0)
breaks_by_horizon <- lapply(seq_along(horizons), function(i) {
  all_x <- unlist(errors_by_horizon[[i]], use.names = FALSE)
  width <- bin_widths[i]
  seq(floor(min(all_x) / width) * width,
      ceiling(max(all_x) / width) * width, by = width)
})

options(repr.plot.width = 15, repr.plot.height = 11)
old_par <- par(no.readonly = TRUE)
par(bg = "white", mfrow = c(3, 3), mar = c(3.8, 4.1, 2.8, 0.8))
for (i in seq_along(horizons)) {
  common_breaks <- breaks_by_horizon[[i]]
  densities <- lapply(errors_by_horizon[[i]], function(x) {
    hist(x, breaks = common_breaks, plot = FALSE)$density
  })
  y_max <- max(unlist(densities)) * 1.12
  for (j in seq_along(period_labels)) {
    x <- errors_by_horizon[[i]][[period_labels[j]]]
    hist(x, breaks = common_breaks, freq = FALSE,
         xlim = range(common_breaks), ylim = c(0, y_max),
         col = period_colors[j], border = "white",
         xlab = "Signed error (p.p.)",
         ylab = if (j == 1L) "Density" else "",
         main = paste0(horizons[i], "m | ", period_labels[j]))
    abline(v = 0, col = "black", lty = 2, lwd = 1.4)
  }
}
par(old_par)

![Signed Focus error histograms by horizon and historical period](figures/ipca_forecast_error_period_histograms.png)

## 4. Compare empirical cumulative distributions

For each horizon, plot the share of forecast months with signed error at or below each threshold. The ECDF value at zero is the share of forecasts with nonpositive error. The table below also reports the largest absolute separation between each pair of period ECDFs; these are **descriptive distances**, not significance tests.

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 12)
old_par <- par(no.readonly = TRUE)
par(bg = "white", mfrow = c(3, 1), mar = c(4.2, 4.5, 2.7, 1))
ecdf_distances <- list()
for (i in seq_along(horizons)) {
  period_errors <- errors_by_horizon[[i]]
  all_x <- unlist(period_errors, use.names = FALSE)
  plot(ecdf(period_errors[[1]]), do.points = FALSE, verticals = TRUE,
       xlim = range(all_x), col = period_colors[1], lwd = 2,
       xlab = "Signed error (p.p.)",
       ylab = "Share at or below x",
       main = paste0(horizons[i], "-month forecast errors"))
  for (j in 2:3) {
    lines(ecdf(period_errors[[j]]), do.points = FALSE, verticals = TRUE,
          col = period_colors[j], lwd = 2)
  }
  abline(v = 0, col = "grey40", lty = 2)
  legend("bottomright", period_labels, col = period_colors,
         lty = 1, lwd = 2, bty = "n", cex = 0.9)
  grid <- sort(unique(all_x))
  for (pair in combn(seq_along(period_labels), 2, simplify = FALSE)) {
    ecdf_distances[[length(ecdf_distances) + 1L]] <- data.frame(
      horizon_months = horizons[i], first_period = period_labels[pair[1]],
      second_period = period_labels[pair[2]],
      max_ecdf_gap = max(abs(ecdf(period_errors[[pair[1]]])(grid) -
                             ecdf(period_errors[[pair[2]]])(grid))))
  }
}
par(old_par)
print(do.call(rbind, ecdf_distances), row.names = FALSE, digits = 4)

![Empirical CDFs of signed Focus errors by horizon and historical period](figures/ipca_forecast_error_period_ecdfs.png)

## 5. Track center, tails, and spread through time

Compute a trailing 36-forecast-month median, 10th and 90th percentile, and standard deviation for each horizon. Each window includes three complete cycles of calendar months. The shaded band tracks the middle 80% of signed errors; the right-hand plots track their standard deviation. Windows end at the forecast month shown on the x-axis.

In [ ]:
window_months <- 36L
window_ends <- seq.int(window_months, nrow(panel))
rolling_by_horizon <- lapply(seq_along(horizons), function(i) {
  values <- panel[[error_columns[i]]]
  stats <- t(vapply(window_ends, function(last_month) {
    x <- values[(last_month - window_months + 1L):last_month]
    c(p10 = unname(quantile(x, 0.10)), median = median(x),
      p90 = unname(quantile(x, 0.90)), sd = sd(x))
  }, numeric(4)))
  data.frame(date = panel$date[window_ends], stats)
})
stopifnot(all(vapply(rolling_by_horizon, nrow, integer(1)) == 244L),
          all(vapply(rolling_by_horizon, function(x) {
            all(is.finite(as.matrix(x[-1])))
          }, logical(1))))

options(repr.plot.width = 15, repr.plot.height = 11)
old_par <- par(no.readonly = TRUE)
par(bg = "white", mfrow = c(3, 2), mar = c(3.8, 4.6, 2.9, 0.8))
for (i in seq_along(horizons)) {
  rolling <- rolling_by_horizon[[i]]
  shade <- adjustcolor(period_colors[i], alpha.f = 0.25)
  plot(rolling$date, rolling$median, type = "n",
       ylim = range(c(rolling$p10, rolling$p90)),
       xlab = "Window end", ylab = "Signed error (p.p.)",
       main = paste0(horizons[i], "m: rolling center and tails"))
  polygon(c(rolling$date, rev(rolling$date)),
          c(rolling$p10, rev(rolling$p90)), col = shade, border = NA)
  lines(rolling$date, rolling$median, col = period_colors[i], lwd = 2)
  abline(h = 0, col = "grey45", lty = 2)

  plot(rolling$date, rolling$sd, type = "l", col = period_colors[i], lwd = 2,
       ylim = c(0, max(rolling$sd) * 1.05),
       xlab = "Window end", ylab = "SD (p.p.)",
       main = paste0(horizons[i], "m: rolling spread"))
}
par(old_par)

![Rolling 36-month quantiles and standard deviations of signed Focus errors](figures/ipca_forecast_error_rolling_distribution.png)

## Interpretation and limits

The complete-year period summaries show negative average signed errors at every horizon and a smaller error spread in 2010–2017 than in the earlier or later period. The rolling plots show *when* the center and spread moved. These are changes in forecast performance, which can reflect changes in realized inflation, forecast behavior, or both; the notebook does not isolate a cause.

Compare periods **within** each horizon. Larger raw 12-month errors are expected because the forecasted cumulative inflation window is longer. Adjacent forecast errors share realized months (up to 2, 5, or 11 months for the 3-, 6-, and 12-month horizons), and the rolling windows overlap. Therefore the months are not independent draws. The ECDF gaps and charts are descriptive; no independent-observation p-values are reported. The three period boundaries and 36-month window are analysis choices, so check robustness before making a stronger claim.